<a id="setup"></a>
# SentencePiece — Option B : Schéma BIO

Le premier sous-token d'un mot reçoit B-TAG (begin), les suivants I-TAG (inside) — schéma standard en extraction d'information. Le tagset double de taille (<PAD> + B/I pour chacun des 11 tags = 23 classes), mais le modèle sait explicitement où un mot commence.

In [8]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))
from bambara_pos_utils import *

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

SP_DIR = Path.cwd().parent  # 07_sentencepiece_tokenization/

Device : cpu


In [2]:
corpus_path = Path.cwd().parent.parent / "04_bambara_pos_pipeline" / "bambara_pos_prep_retagged.conll"
corpus_data = load_bambara_corpus(corpus_path)
train_data, val_data, test_data = split_corpus(corpus_data, seed=42)
print(f"Train : {len(train_data)} | Val : {len(val_data)} | Test : {len(test_data)}")

Corpus chargé : 37392 phrases.
Train : 29913 | Val : 3739 | Test : 3740


In [3]:
# Réutilisation du même modèle SentencePiece que l'option A (déjà entraîné)
sp_path = SP_DIR / "bambara_sp.model"
if not sp_path.is_file():
    raise FileNotFoundError(
        "bambara_sp.model introuvable : exécute d'abord 07a_broadcast_scheme "
        "pour entraîner le modèle SentencePiece partagé."
    )

sp = spm.SentencePieceProcessor(model_file=str(sp_path))
print("Taille du vocabulaire de sous-mots :", sp.get_piece_size())

Taille du vocabulaire de sous-mots : 4000


<a id="bio-tagset"></a>
# Construction du tagset BIO

In [4]:
label2id_bio, id2label_bio = build_bio_tagset(VALID_TAGS)
print(f"Taille du tagset BIO : {len(label2id_bio)}")
for tag, idx in label2id_bio.items():
    print(f"  {tag:<10} : {idx}")

Taille du tagset BIO : 23
  <PAD>      : 0
  B-ADJ      : 1
  I-ADJ      : 2
  B-ADV      : 3
  I-ADV      : 4
  B-AUX      : 5
  I-AUX      : 6
  B-CONJ     : 7
  I-CONJ     : 8
  B-DET      : 9
  I-DET      : 10
  B-NOM      : 11
  I-NOM      : 12
  B-PART     : 13
  I-PART     : 14
  B-POSTP    : 15
  I-POSTP    : 16
  B-PRON     : 17
  I-PRON     : 18
  B-PUNCT    : 19
  I-PUNCT    : 20
  B-VERBE    : 21
  I-VERBE    : 22


In [5]:
# Vérification de l'alignement sur une phrase du corpus
words, tags = train_data[0]
subword_tokens, subword_tags = encode_sentence_with_tags(words, tags, sp, scheme="bio")

print("Mots originaux :", words)
print("Tags originaux :", tags)
print()
for tok, tag in zip(subword_tokens, subword_tags):
    print(f"  {tok:<12} {tag}")

Mots originaux : ['Ani', 'ne', 'ye', 'buurukun', 'wolonwula', 'kari', 'kari', 'ka', 'di', 'mɔgɔ', 'ba', 'naani', 'ma', 'tuma', 'min', ',', 'aw', 'ye', 'segi', 'joli', 'fa', 'kunkuruw', 'tɔ', 'la', '?']
Tags originaux : ['CONJ', 'PRON', 'AUX', 'VERBE', 'NOM', 'NOM', 'NOM', 'AUX', 'VERBE', 'NOM', 'NOM', 'NOM', 'AUX', 'NOM', 'PRON', 'PUNCT', 'PRON', 'AUX', 'VERBE', 'NOM', 'NOM', 'NOM', 'NOM', 'POSTP', 'PUNCT']

  ▁Ani         B-CONJ
  ▁ne          B-PRON
  ▁ye          B-AUX
  ▁buurukun    B-VERBE
  ▁wolonwula   B-NOM
  ▁kari        B-NOM
  ▁kari        B-NOM
  ▁ka          B-AUX
  ▁di          B-VERBE
  ▁mɔgɔ        B-NOM
  ▁ba          B-NOM
  ▁naani       B-NOM
  ▁ma          B-AUX
  ▁tuma        B-NOM
  ▁min         B-PRON
  ▁,           B-PUNCT
  ▁aw          B-PRON
  ▁ye          B-AUX
  ▁segi        B-VERBE
  ▁joli        B-NOM
  ▁fa          B-NOM
  ▁kun         B-NOM
  kuru         I-NOM
  w            I-NOM
  ▁tɔ          B-NOM
  ▁la          B-POSTP
  ▁?           B-PUNCT


<a id="training"></a>
# Entraînement du modèle BiLSTM sur le tagset BIO

In [6]:
hyperparams = {
    "learning_rate": 1e-3,
    "batch_size": 32,
    "hidden_dim": 128,
    "embedding_dim": 100,
}

MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)
checkpoint_path = MODELS_DIR / "bambara_pos_sp_bio.pth"

model, history, best_val_loss = run_training(
    train_data, val_data,
    dataset_cls=BambaraSubwordDataset,
    dataset_args=(sp, label2id_bio, "bio"),
    vocab_size=sp.get_piece_size(),
    tagset_size=len(label2id_bio),
    hyperparams=hyperparams,
    device=device,
    patience=4, max_epochs=30,
    checkpoint_path=checkpoint_path,
)

Époque  1 | Train Loss : 0.2227 | Val Loss : 0.0250 | Val Acc : 99.40%
Époque  2 | Train Loss : 0.0102 | Val Loss : 0.0074 | Val Acc : 99.87%
Époque  3 | Train Loss : 0.0025 | Val Loss : 0.0035 | Val Acc : 99.91%
Époque  4 | Train Loss : 0.0010 | Val Loss : 0.0031 | Val Acc : 99.92%
Époque  5 | Train Loss : 0.0005 | Val Loss : 0.0020 | Val Acc : 99.94%
Époque  6 | Train Loss : 0.0005 | Val Loss : 0.0021 | Val Acc : 99.95%
Époque  7 | Train Loss : 0.0001 | Val Loss : 0.0015 | Val Acc : 99.95%
Époque  8 | Train Loss : 0.0001 | Val Loss : 0.0048 | Val Acc : 99.87%
Époque  9 | Train Loss : 0.0003 | Val Loss : 0.0029 | Val Acc : 99.94%
Époque 10 | Train Loss : 0.0001 | Val Loss : 0.0011 | Val Acc : 99.96%
Époque 11 | Train Loss : 0.0000 | Val Loss : 0.0016 | Val Acc : 99.95%
Époque 12 | Train Loss : 0.0002 | Val Loss : 0.0011 | Val Acc : 99.97%
Époque 13 | Train Loss : 0.0000 | Val Loss : 0.0010 | Val Acc : 99.98%
Époque 14 | Train Loss : 0.0000 | Val Loss : 0.0011 | Val Acc : 99.97%
Époque

<a id="evaluation"></a>
# Évaluation au niveau mot

In [10]:
def evaluate_subword_model_word_level(model, sentences, sp, id2label, scheme, device):
    """
    Évalue un modèle entraîné sur des sous-tokens SentencePiece, au niveau mot.
    Reconstruit un tag par mot à partir des prédictions sous-token avant de comparer
    aux vrais tags, pour rester comparable aux évaluations mot-entier (05/06).
    """
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for words, tags in sentences:
            subword_tokens, _ = encode_sentence_with_tags(words, tags, sp, scheme)
            if not subword_tokens:
                continue
            ids = [sp.piece_to_id(t) for t in subword_tokens]
            input_tensor = torch.tensor([ids], dtype=torch.long).to(device)
            preds = torch.argmax(model(input_tensor), dim=-1).squeeze(0).cpu().tolist()
            predicted_tags = [id2label[p] for p in preds]

            reconstructed = reconstruct_word_tags(subword_tokens, predicted_tags, scheme)
            n = min(len(reconstructed), len(tags))
            all_true.extend(tags[:n])
            all_pred.extend(reconstructed[:n])

    accuracy = accuracy_score(all_true, all_pred)
    report = classification_report(all_true, all_pred, zero_division=0)
    return accuracy, report

In [12]:
test_acc, test_report = evaluate_subword_model_word_level(
    model, test_data, sp, id2label_bio, "bio", device 
)
print(f"Accuracy au niveau mot (SentencePiece, BIO) : {test_acc*100:.2f}%\n")
print(test_report)

Accuracy au niveau mot (SentencePiece, BIO) : 99.97%

              precision    recall  f1-score   support

         ADJ       0.99      1.00      1.00       251
         ADV       1.00      1.00      1.00       252
         AUX       1.00      1.00      1.00      9650
        CONJ       1.00      1.00      1.00      2812
         DET       1.00      1.00      1.00      2516
         NOM       1.00      1.00      1.00     18577
        PART       1.00      1.00      1.00       841
       POSTP       1.00      1.00      1.00      3411
        PRON       1.00      1.00      1.00      9114
       PUNCT       1.00      1.00      1.00     10488
       VERBE       1.00      1.00      1.00      6444

    accuracy                           1.00     64356
   macro avg       1.00      1.00      1.00     64356
weighted avg       1.00      1.00      1.00     64356



## **Interprétation**

Compare ce score à celui de l'option A (broadcasting) et à celui du module 06 (mot-entier). Si le BIO n'apporte pas de gain net malgré la complexité supplémentaire (tagset doublé), le broadcasting reste le meilleur compromis simplicité/performance pour ce projet — un résultat négatif est une conclusion valable et utile à documenter, pas un échec.